# COMI-LINGUA - Difficulty Assignment (POS tagging)

Assigns `difficulty` (Easy / Medium / Hard) to 300 sampled COMI-LINGUA rows.

**Task.** Hindi-English code-mixed POS tagging. `question` is the sentence,
`answer` is Annotator 1's gold tag sequence, `eval_metric` is `accuracy`. There
are no options - the model must *generate* a tag per token.

**Method.** Same 3-model voting protocol as the other splits, with one change
forced by the task. A sequence task has no single right/wrong answer, so each
model's output is scored by **token accuracy**, and a model "passes" a row when
its accuracy clears `TAG_ACC_THRESHOLD`. Those three pass/fail votes are then
summed exactly as before:

| Models passing | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

Raw accuracies are stored in the progress files, so you can re-threshold in
Cell 9 **without re-running any model**.


**Output.** `comi_lingua_difficulty.jsonl` - the original 14 schema fields with
`difficulty` filled in, plus an audit file holding each model's tags.

### Cell 1 - Install dependencies and authenticate

Installs the HuggingFace stack, then logs in.

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; only Mistral is
open. Accept the licence on each model page, create a **read** token at
huggingface.co/settings/tokens, then in Colab add it via the **key icon** in the
left sidebar as a secret named `HF_TOKEN` with notebook access enabled.

Use the secret rather than pasting the token into a cell - a pasted token is
saved inside the notebook file.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 40.9 MB/s eta 0:00:00
HF login OK


### Cell 2 - Mount Drive

Drive is the weight cache: models are downloaded once, quantised to 4-bit, and
saved here so later runs skip the download entirely.

`DRIVE_OK` records whether the mount actually worked - later cells check it
rather than assuming. If you see **"credential propagation was unsuccessful"**,
the auth popup did not complete. Re-run and finish the popup; allow pop-ups and
third-party cookies for `colab.research.google.com`; or mount from the Files
sidebar. Without Drive everything still runs, but nothing is cached.

In [2]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

Mounted at /drive
Drive mounted | weight cache: /drive/MyDrive/models


### Cell 3 - Configuration

- `ALIGNABLE_ONLY` - sample only rows whose rebuilt tokenisation matches the
  gold tag count exactly. Leave `True`: on the other ~16% of rows a positional
  accuracy would be comparing tags against the wrong tokens. The trade-off is
  that the sample is no longer uniform over the whole file - it skews slightly
  toward cleanly-tokenising sentences.
- `TAG_ACC_THRESHOLD` - token accuracy a model must reach to "pass" a row. 0.8
  is a reasonable default for a 14-tag set. Changing it does **not** require
  re-running the models: raw accuracies are stored, and Cell 9 re-thresholds.
- `MAX_NEW_TOKENS` - hard cap on generation length per row.
- `MODELS` - three judges from three different families (Mistral / Meta /
  Google) so their errors decorrelate. `USE_INSTRUCT` picks instruction-tuned
  checkpoints, which are much better at following a tagging format; Cell 5
  detects base vs chat automatically either way.

In [3]:
import gc
import re
import json
import random
import shutil
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- paths ----
INPUT_FILE  = "comi_lingua.jsonl"
OUTPUT_FILE = "comi_lingua_difficulty.jsonl"
AUDIT_FILE  = "comi_lingua_audit.jsonl"
PROG_DIR    = "judge_progress"

# ---- sampling ----
N_ROWS         = 200
SEED           = 42
ALIGNABLE_ONLY = True

# ---- scoring ----
TAG_ACC_THRESHOLD = 0.80      # token accuracy needed to "pass" a row
MAX_NEW_TOKENS    = 512

# ---- batching ----
BATCH_SIZE = 25

# ---- the three judges ----
USE_INSTRUCT = True

REPOS = {
    True: {
        "mistral": "mistralai/Mistral-7B-Instruct-v0.3",   # ungated
        "llama":   "meta-llama/Llama-3.1-8B-Instruct",     # GATED
        "gemma":   "google/gemma-2-9b-it",                 # GATED
    },
    False: {
        "mistral": "mistralai/Mistral-7B-v0.3",
        "llama":   "meta-llama/Llama-3.1-8B",
        "gemma":   "google/gemma-2-9b",
    },
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)
print("Judges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached in Drive" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

Judges (instruct):
  mistral  mistralai/Mistral-7B-Instruct-v0.3         cached in Drive
  llama    meta-llama/Llama-3.1-8B-Instruct           cached in Drive
  gemma    google/gemma-2-9b-it                       cached in Drive

device: cuda


### Cell 4 - Load, tokenise, and sample

The file stores only the raw sentence, so the token boundaries the annotator
used have to be reconstructed before per-token accuracy means anything.

`tokenize()` is a Devanagari-aware regex with one crucial detail: the danda
`.` (U+0964) and double danda (U+0965) sit *inside* the Devanagari Unicode
block, so a naive `[\u0900-\u097F]+` class glues sentence-final punctuation
onto the last word (`hain.` as one token) and every such row is off by one.
Excluding them lifts the exact-match rate from 42% to **83.8%**. Hashtags,
@mentions and URLs are kept whole, since the annotators treated them as single
tokens.

Rows are then filtered to those where token count equals gold tag count, and
200 are sampled from that pool under a fixed seed - so the same 200 come back
on every run, which is what makes the resume logic safe.

In [4]:
with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))

# Devanagari block MINUS danda (U+0964) and double danda (U+0965), which are
# punctuation and must not be glued onto the preceding word.
_DEV = "\u0900-\u0963\u0966-\u097F"
_TOK = re.compile(
    r"https?://\S+|[@#][\w{d}]+|[\w{d}]+|[^\s\w{d}]".format(d=_DEV),
    re.UNICODE)


def tokenize(sentence):
    return _TOK.findall(sentence)


TAGSET = sorted({t for r in all_rows for t in r["answer"].split()})
print("tag set ({}): {}".format(len(TAGSET), " ".join(TAGSET)))

# keep only rows where our tokenisation lines up 1:1 with the gold tags
for r in all_rows:
    r["_tokens"] = tokenize(r["question"])
    r["_gold"]   = r["answer"].split()
    r["_ok"]     = len(r["_tokens"]) == len(r["_gold"])

alignable = [r for r in all_rows if r["_ok"]]
print("alignable rows: {}/{} ({:.1%})".format(
    len(alignable), len(all_rows), len(alignable) / len(all_rows)))

pool = alignable if ALIGNABLE_ONLY else all_rows
assert len(pool) >= N_ROWS, "not enough rows to sample from"

random.seed(SEED)
sample = random.sample(pool, N_ROWS)

lens = sorted(len(r["_tokens"]) for r in sample)
print("\nSampled {} rows | tokens per row: min {}, median {}, max {}".format(
    len(sample), lens[0], lens[len(lens) // 2], lens[-1]))
print("tag distribution in sample:",
      Counter(t for r in sample for t in r["_gold"]).most_common(6))

print("\n--- alignment check on the first sampled row ---")
r = sample[0]
print("  " + " ".join("{}/{}".format(a, b) for a, b in zip(r["_tokens"], r["_gold"])))

Loaded 1999 rows from comi_lingua.jsonl
tag set (14): ADJ ADP ADV CONJ DET NOUN NUM PART PART_NEG PRON PRON_WH PROPN VERB X
alignable rows: 1676/1999 (83.8%)

Sampled 200 rows | tokens per row: min 2, median 21, max 43
tag distribution in sample: [('NOUN', 937), ('VERB', 744), ('ADP', 643), ('X', 551), ('PROPN', 533), ('ADJ', 238)]

--- alignment check on the first sampled row ---
  पाकिस्तानी/ADJ क्रिकेटर/NOUN कामरान/PROPN अकमल/PROPN (/X Kamran/PROPN Akmal/PROPN )/X ने/ADP इंडिया/PROPN vs/PART इंग्लैंड/PROPN (/X IND/PROPN vs/PART ENG/PROPN )/X नॉटिंघम/PROPN टेस्ट/NOUN मैच/NOUN को/ADP लेकर/VERB बड़ी/ADJ प्रतिक्रिया/VERB दी/VERB है/VERB ।/X


### Cell 5 - Build the tagging prompt

The model is shown the **pre-tokenised** sentence and asked for one tag per
token, space separated. Giving it our tokenisation is what keeps its output
positionally comparable to the gold tags - asking it to tag raw text would let
it choose its own token boundaries and silently destroy the alignment.

Few-shot examples are drawn from alignable rows **outside** the 200-row sample,
so no scored row ever has its gold tags shown to the model. They are picked
deterministically from `SEED`, so every model and every re-run sees the same
examples.

Two builders, as before: `build_completion` (flat text, for base checkpoints,
ending on a dangling `Tags:`) and `build_chat_messages` (chat turns, for
instruct checkpoints). Cell 6 picks per model.

In [5]:
INSTRUCTIONS = (
    "You are a part-of-speech tagger for Hindi-English code-mixed text "
    "(Hinglish), written in Devanagari or Roman script.\n\n"
    "You are given a sentence already split into tokens. Output exactly one "
    "tag per token, in the same order, separated by single spaces.\n\n"
    "Use only these tags:\n" + " ".join(TAGSET) + "\n\n"
    "Output the tag sequence and nothing else - no numbering, no explanation, "
    "no repetition of the tokens."
)


def pick_fewshot(k=3):
    # short, cleanly aligned rows from OUTSIDE the sample
    used = {r["id"] for r in sample}
    pool = [r for r in alignable
            if r["id"] not in used and 6 <= len(r["_tokens"]) <= 14]
    random.Random(SEED + 1).shuffle(pool)
    return pool[:k]


FEWSHOT = pick_fewshot()


def build_completion(tokens):
    text = INSTRUCTIONS + "\n"
    for r in FEWSHOT:
        text += "\nTokens: {}\nTags: {}\n".format(
            " ".join(r["_tokens"]), " ".join(r["_gold"]))
    text += "\nTokens: {}\nTags:".format(" ".join(tokens))
    return text


def build_chat_messages(tokens):
    msgs = [{"role": "system", "content": INSTRUCTIONS}]
    for r in FEWSHOT:
        msgs.append({"role": "user",
                     "content": "Tokens: " + " ".join(r["_tokens"])})
        msgs.append({"role": "assistant", "content": " ".join(r["_gold"])})
    msgs.append({"role": "user", "content": "Tokens: " + " ".join(tokens)})
    return msgs


print("Few-shot examples ({}), all from OUTSIDE the sample:".format(len(FEWSHOT)))
for r in FEWSHOT:
    print("  {} ({} tokens)".format(r["id"], len(r["_tokens"])))

print("\n" + "=" * 64)
print(build_completion(sample[0]["_tokens"]))
print("=" * 64)
print("[gold: {}]".format(" ".join(sample[0]["_gold"])))

Few-shot examples (3), all from OUTSIDE the sample:
  comi_005913 (9 tokens)
  comi_000521 (14 tokens)
  comi_004584 (12 tokens)

You are a part-of-speech tagger for Hindi-English code-mixed text (Hinglish), written in Devanagari or Roman script.

You are given a sentence already split into tokens. Output exactly one tag per token, in the same order, separated by single spaces.

Use only these tags:
ADJ ADP ADV CONJ DET NOUN NUM PART PART_NEG PRON PRON_WH PROPN VERB X

Output the tag sequence and nothing else - no numbering, no explanation, no repetition of the tokens.

Tokens: shaan ka bag radio lag rha hai : v
Tags: PROPN ADP NOUN NOUN VERB VERB VERB X X

Tokens: Bihar के ड्रामा और एक्टिंग स्टूडेंट्स को लेकर क्या बोले Manoj Bajpayee ? #ManojBajpayee
Tags: PROPN ADP NOUN CONJ NOUN NOUN ADP VERB PRON_WH VERB PROPN PROPN X PROPN

Tokens: surgical strike to vote ke liye huaa karna hai to aab kijiye
Tags: ADJ NOUN PART NOUN ADP ADP VERB VERB VERB PART ADV VERB

Tokens: पाकिस्तानी क्रिकेटर

### Cell 6 - Generate tags and score them

Unlike a multiple-choice task, this one has to **generate**. `predict_tags`
decodes greedily (`do_sample=False`) so results are reproducible, with
`max_new_tokens` scaled to the token count rather than fixed, capped by
`MAX_NEW_TOKENS`.

`parse_tags` keeps only strings that are actually in `TAGSET`, reading just the
first line of output. That discards stray prose without needing the model to be
perfectly obedient - a tag it invents is simply dropped rather than counted.

`tag_accuracy` divides matches by `max(len(pred), len(gold))`, so both
under-generating and over-generating are penalised. A model that emits three
tags for a twenty-token sentence cannot score well by getting those three
right.

In [6]:
_TAGRE = re.compile(r"[A-Z][A-Z_]*")


def parse_tags(text):
    # keep only real tags from the first line; drop anything invented
    line = text.strip().split("\n")[0] if text.strip() else ""
    return [t for t in _TAGRE.findall(line) if t in TAGSET]


@torch.no_grad()
def predict_tags(model, tokenizer, tokens):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(tokens)
    else:
        msgs = build_chat_messages(tokens)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    budget = min(6 * len(tokens) + 16, MAX_NEW_TOKENS)

    out = model.generate(**inputs,
                         max_new_tokens=budget,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return parse_tags(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


def tag_accuracy(pred, gold):
    # positional accuracy, penalising length mismatch in both directions
    if not gold:
        return 0.0
    n = min(len(pred), len(gold))
    matches = sum(p == g for p, g in zip(pred[:n], gold[:n]))
    return matches / max(len(pred), len(gold))


def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()

print("Generation and scoring functions defined")

Generation and scoring functions defined


### Cell 7 - Load-or-cache, and the batched runner

`load_model` implements download-once: if `models/<name>_4bit` exists in Drive
it is loaded directly (already 4-bit, so passing a fresh `BitsAndBytesConfig`
would conflict and is omitted); otherwise the repo is downloaded, quantised,
and saved to Drive for next time. `trust_remote_code` stays off - repo-shipped
modelling code is often written against an older transformers API.

`run_model` stores the **raw accuracy** for every row, not a pass/fail. That is
deliberate: the threshold is applied later in Cell 9, so you can tighten or
loosen `TAG_ACC_THRESHOLD` and re-derive difficulty without spending another
GPU-hour. Each finished batch is appended to
`judge_progress/<model>.jsonl` before the next begins, so a disconnect costs at
most `BATCH_SIZE` rows.

In [7]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog_file = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    # ---- 1. resume from whatever is already on disk ----
    done = {}
    if os.path.exists(prog_file):
        with open(prog_file, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already scored".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    # ---- 2. load (from Drive if cached, else download and cache) ----
    model, tokenizer = load_model(spec)

    # ---- 3. score in batches, saving after each one ----
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_start in range(0, len(remaining), BATCH_SIZE):
        batch     = remaining[batch_start : batch_start + BATCH_SIZE]
        batch_num = batch_start // BATCH_SIZE + 1

        batch_results = []
        for row in batch:
            pred = predict_tags(model, tokenizer, row["_tokens"])
            batch_results.append({
                "id":        row["id"],
                "accuracy":  tag_accuracy(pred, row["_gold"]),
                "n_pred":    len(pred),
                "n_gold":    len(row["_gold"]),
                "predicted": " ".join(pred),
            })

        with open(prog_file, "a", encoding="utf-8") as f:
            for item in batch_results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        done.update({item["id"]: item for item in batch_results})
        mean_acc = sum(v["accuracy"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | mean token acc {:.1%}".format(
            batch_num, total_batches, len(done), len(rows), mean_acc))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

Runner defined


### Cell 8 - Run all three models

One model at a time - loaded, scored, unloaded - so peak VRAM stays near 6 GB
instead of the ~17 GB all three would need together.

This is the long cell, and slower than a multiple-choice run because every row
generates a full tag sequence rather than a single token. Budget roughly
**8-12 min per model** for inference, plus downloads on the first run. Safe to
re-run: anything already scored is skipped.

In [8]:
preds = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    preds[spec["name"]] = run_model(spec, sample)

print("\nAll models done")


=== mistral ===
  loading /drive/MyDrive/models/mistral_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 4.14GB | prompt style: chat
  batch 1/8 saved - 25/200 rows | mean token acc 13.0%
  batch 2/8 saved - 50/200 rows | mean token acc 9.4%
  batch 3/8 saved - 75/200 rows | mean token acc 11.5%
  batch 4/8 saved - 100/200 rows | mean token acc 11.0%
  batch 5/8 saved - 125/200 rows | mean token acc 10.8%
  batch 6/8 saved - 150/200 rows | mean token acc 10.2%
  batch 7/8 saved - 175/200 rows | mean token acc 10.4%
  batch 8/8 saved - 200/200 rows | mean token acc 10.9%
  mistral complete

=== llama ===
  loading /drive/MyDrive/models/llama_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 5.71GB | prompt style: chat


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  batch 1/8 saved - 25/200 rows | mean token acc 17.1%
  batch 2/8 saved - 50/200 rows | mean token acc 16.9%
  batch 3/8 saved - 75/200 rows | mean token acc 18.2%
  batch 4/8 saved - 100/200 rows | mean token acc 18.9%
  batch 5/8 saved - 125/200 rows | mean token acc 19.1%
  batch 6/8 saved - 150/200 rows | mean token acc 18.9%
  batch 7/8 saved - 175/200 rows | mean token acc 18.9%
  batch 8/8 saved - 200/200 rows | mean token acc 19.4%
  llama complete

=== gemma ===
  loading /drive/MyDrive/models/gemma_4bit [Drive cache (already 4-bit), attn=eager]


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

  ready | VRAM: 6.14GB | prompt style: chat
  batch 1/8 saved - 25/200 rows | mean token acc 20.8%
  batch 2/8 saved - 50/200 rows | mean token acc 19.8%
  batch 3/8 saved - 75/200 rows | mean token acc 21.8%
  batch 4/8 saved - 100/200 rows | mean token acc 22.4%
  batch 5/8 saved - 125/200 rows | mean token acc 23.3%
  batch 6/8 saved - 150/200 rows | mean token acc 22.9%
  batch 7/8 saved - 175/200 rows | mean token acc 22.7%
  batch 8/8 saved - 200/200 rows | mean token acc 22.8%
  gemma complete

All models done


### Cell 9 - Assign difficulty into the schema

Applies `TAG_ACC_THRESHOLD` to each stored accuracy to get three pass/fail
votes, sums them, and maps the total to Easy / Medium / Hard - the same voting
rule as every other split.

Because the raw accuracies live in the progress files, **re-running just this
cell with a different threshold re-derives difficulty for free.** The printed
sensitivity table shows what other thresholds would have produced, so you can
see whether your split is stable or balanced on a knife edge.

Each output row is rebuilt key-by-key from `SCHEMA_KEYS`, so the file carries
exactly the 14 IndicSample fields in schema order - `difficulty` is the only
value that changes, and the working fields (`_tokens`, `_gold`, `_ok`) never
leak in. Per-model tags go to the audit file.

In [9]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results = []
audit = []

for row in sample:
    accs  = [preds[s["name"]][row["id"]]["accuracy"] for s in MODELS]
    votes = [int(a >= TAG_ACC_THRESHOLD) for a in accs]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":          row["id"],
        "difficulty":  difficulty,
        "votes":       votes,
        "accuracies":  [round(a, 4) for a in accs],
        "threshold":   TAG_ACC_THRESHOLD,
        "n_tokens":    len(row["_gold"]),
        "gold":        " ".join(row["_gold"]),
        "predictions": {s["name"]: preds[s["name"]][row["id"]]["predicted"]
                        for s in MODELS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))

print("\nThreshold sensitivity (no re-running needed):")
print("  {:>9}  {:>6} {:>7} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
for th in [0.6, 0.7, 0.75, 0.8, 0.85, 0.9]:
    d = Counter(get_difficulty([int(preds[s["name"]][r["id"]]["accuracy"] >= th)
                                for s in MODELS]) for r in sample)
    mark = "  <- current" if abs(th - TAG_ACC_THRESHOLD) < 1e-9 else ""
    print("  {:>9.2f}  {:>6} {:>7} {:>6}{}".format(
        th, d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0), mark))

Saved -> comi_lingua_difficulty.jsonl (200 rows)
Audit -> comi_lingua_audit.jsonl

Threshold sensitivity (no re-running needed):
  threshold    Easy  Medium   Hard
       0.60       0       0    200
       0.70       0       0    200
       0.75       0       0    200
       0.80       0       0    200  <- current
       0.85       0       0    200
       0.90       0       0    200


### Cell 10 - Verify and report

Checks before trusting the file:

1. **Schema** - all 14 keys in order, no nulls in `difficulty`, and none of the
   internal `_tokens` / `_gold` / `_ok` fields leaked through.
2. **Difficulty distribution.**
3. **Per-model mean token accuracy**, against two reference points: the
   *most-frequent-tag* baseline (tag everything `NOUN`), and a perfect tagger.
   A model at or below the baseline is not tagging, it is guessing.
4. **Length discipline** - how often each model returned exactly the requested
   number of tags. Chronic under- or over-generation means the prompt is being
   ignored, and accuracy will be low for a formatting reason rather than a
   linguistic one. Check this before concluding a model is bad at Hindi.

In [10]:
# 1. schema integrity
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
leaked   = [r["id"] for r in final_results
            if any(k.startswith("_") for k in r.keys())]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {} | leaked: {}".format(
    len(final_results), len(bad_keys), len(missing), len(leaked)))

# 2. difficulty distribution
dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution (threshold {:.2f}):".format(TAG_ACC_THRESHOLD))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

# 3. per-model accuracy vs. a trivial baseline
gold_tags = [t for r in sample for t in r["_gold"]]
top_tag, top_n = Counter(gold_tags).most_common(1)[0]
print("\nMean token accuracy:")
for s in MODELS:
    acc = sum(v["accuracy"] for v in preds[s["name"]].values()) / total
    print("  {:<10} {:.1%}".format(s["name"], acc))
print("  {:<10} {:.1%}  <- tag-everything-'{}' baseline".format(
    "baseline", top_n / len(gold_tags), top_tag))

# 4. did the models respect the token count?
print("\nLength discipline (exact tag count returned):")
for s in MODELS:
    v = list(preds[s["name"]].values())
    exact = sum(1 for x in v if x["n_pred"] == x["n_gold"])
    short = sum(1 for x in v if x["n_pred"] < x["n_gold"])
    flag  = "  <- ignoring the format, not the language" if exact / total < 0.5 else ""
    print("  {:<10} exact {:>3}/{}  short {:>3}  long {:>3}{}".format(
        s["name"], exact, total, short, total - exact - short, flag))

print("\nSample rows:")
for a in audit[:3]:
    print("  {} | {:<6} | acc {} | {} tokens".format(
        a["id"], a["difficulty"], a["accuracies"], a["n_tokens"]))

Schema check : 200 rows | wrong keys: 0 | null difficulty: 0 | leaked: 0

Difficulty distribution (threshold 0.80):
  Easy   :    0  (0.0%)
  Medium :    0  (0.0%)
  Hard   :  200  (100.0%)

Mean token accuracy:
  mistral    10.9%
  llama      19.4%
  gemma      22.8%
  baseline   22.3%  <- tag-everything-'NOUN' baseline

Length discipline (exact tag count returned):
  mistral    exact  12/200  short 114  long  74  <- ignoring the format, not the language
  llama      exact  18/200  short  82  long 100  <- ignoring the format, not the language
  gemma      exact   7/200  short 190  long   3  <- ignoring the format, not the language

Sample rows:
  comi_012458 | Hard   | acc [0.0741, 0.0674, 0.0741] | 27 tokens
  comi_007788 | Hard   | acc [0.2, 0.3, 0.2] | 10 tokens
  comi_011956 | Hard   | acc [0.2222, 0.3889, 0.2778] | 18 tokens
